In [1]:
import os
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [2]:
load_dotenv("../.env")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [3]:
class QuestionAnswer(BaseModel):
    question: str = Field(description="Interview question")
    answer: str = Field(description="Candidate's answer")


class InterviewAnswers(BaseModel):
    answers: List[QuestionAnswer]

In [4]:
parser_step1 = PydanticOutputParser(
    pydantic_object=InterviewAnswers
)

step1_prompt = PromptTemplate(
    template="""
You are an interview transcript analyzer.

Extract every interview question and the candidate's corresponding answer
from the transcript.

Do not invent information. Use only information available in the transcript.

{format_instructions}

Interview Transcript:
{transcript_text}
""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions": parser_step1.get_format_instructions()
    }
)

step1_chain = step1_prompt | model | parser_step1

In [5]:
transcript_text = """
Interviewer: Tell me about your experience with Python.

Candidate: I have been using Python for about two years. I used
Python with Pandas and NumPy for data analysis and built a few
machine learning projects.

Interviewer: Have you worked with SQL?

Candidate: Yes. I have used MySQL and SQL for querying databases.
I have written SELECT, JOIN, GROUP BY and aggregate queries.

Interviewer: Tell me about your experience with Docker.

Candidate: I have heard about Docker and understand that it is
used for containerization, but I have not personally used it
in a project.

Interviewer: Have you worked with machine learning?

Candidate: Yes. I have trained classification models using
Scikit-learn and evaluated them using accuracy, precision,
recall and F1-score.
"""

skills_to_assess = ["Python", "SQL", "Docker", "Machine Learning"]

In [6]:
step1_result = step1_chain.invoke({
    "transcript_text": transcript_text
})

print(step1_result)

answers=[QuestionAnswer(question='Tell me about your experience with Python.', answer='I have been using Python for about two years. I used Python with Pandas and NumPy for data analysis and built a few machine learning projects.'), QuestionAnswer(question='Have you worked with SQL?', answer='Yes. I have used MySQL and SQL for querying databases. I have written SELECT, JOIN, GROUP BY and aggregate queries.'), QuestionAnswer(question='Tell me about your experience with Docker.', answer='I have heard about Docker and understand that it is used for containerization, but I have not personally used it in a project.'), QuestionAnswer(question='Have you worked with machine learning?', answer='Yes. I have trained classification models using Scikit-learn and evaluated them using accuracy, precision, recall and F1-score.')]


In [7]:
class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being evaluated")
    demonstrated: bool = Field(description="Whether the candidate demonstrated the skill")
    evidence_or_gap: str = Field(description="Evidence or gap")


class SkillEvaluations(BaseModel):
    evaluations: List[SkillEvaluation]

In [8]:
parser_step2 = PydanticOutputParser(
    pydantic_object=SkillEvaluations
)

step2_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a technical interviewer evaluating skill evidence.

Evaluate each requested skill using ONLY the candidate's answers.

A skill should be marked demonstrated=True only when the candidate
provides clear evidence that they possess or have used that skill.

If the skill is not clearly demonstrated, mark demonstrated=False
and explain the gap.

Do not assume skills that are not mentioned.
"""),
    ("human", """
Skills to assess:
{skills_to_assess}

Candidate's extracted answers:
{answers}

{format_instructions}
""")
]).partial(
    format_instructions=parser_step2.get_format_instructions()
)

step2_chain = step2_prompt | model | parser_step2

In [9]:
step2_result = step2_chain.invoke({
    "skills_to_assess": ", ".join(skills_to_assess),
    "answers": step1_result.model_dump_json()
})

print(step2_result)

evaluations=[SkillEvaluation(skill='Python', demonstrated=True, evidence_or_gap='Used Python for about two years, employing Pandas and NumPy for data analysis and building several machine learning projects.'), SkillEvaluation(skill='SQL', demonstrated=True, evidence_or_gap='Worked with MySQL, writing SELECT, JOIN, GROUP BY, and aggregate queries to query databases.'), SkillEvaluation(skill='Docker', demonstrated=False, evidence_or_gap='Has only theoretical knowledge of Docker and has not personally used it in any project.'), SkillEvaluation(skill='Machine Learning', demonstrated=True, evidence_or_gap='Trained classification models using Scikit-learn and evaluated them with metrics such as accuracy, precision, recall, and F1-score.')]


In [10]:
class ScorecardRow(BaseModel):
    skill: str = Field(description="Skill name")
    status: str = Field(description="Demonstrated or Not Demonstrated")
    evidence_or_gap: str = Field(description="Evidence or identified gap")


class FinalScorecard(BaseModel):
    scorecard_rows: List[ScorecardRow]
    overall_recommendation: str = Field(
        description="Overall hiring recommendation"
    )

In [11]:
parser_step3 = PydanticOutputParser(
    pydantic_object=FinalScorecard
)

step3_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a hiring panel coordinator.

Create a structured candidate scorecard from the technical skill evaluation.

Summarize the evidence clearly and provide an overall recommendation.

Base the recommendation only on the supplied evaluation.
Do not invent candidate information.
"""),
    ("human", """
Skill evaluation:

{skill_evaluation}

{format_instructions}
""")
]).partial(
    format_instructions=parser_step3.get_format_instructions()
)

step3_chain = step3_prompt | model | parser_step3

In [12]:
step3_result = step3_chain.invoke({
    "skill_evaluation": step2_result.model_dump_json()
})

for row in step3_result.scorecard_rows:
    print(f"Skill       : {row.skill}")
    print(f"Status      : {row.status}")
    print(f"Evidence/Gap: {row.evidence_or_gap}")
    print("-" * 50)

print(f"\nOverall Recommendation: {step3_result.overall_recommendation}")

Skill       : Python
Status      : Demonstrated
Evidence/Gap: Used Python for about two years, employing Pandas and NumPy for data analysis and building several machine learning projects.
--------------------------------------------------
Skill       : SQL
Status      : Demonstrated
Evidence/Gap: Worked with MySQL, writing SELECT, JOIN, GROUP BY, and aggregate queries to query databases.
--------------------------------------------------
Skill       : Docker
Status      : Not Demonstrated
Evidence/Gap: Has only theoretical knowledge of Docker and has not personally used it in any project.
--------------------------------------------------
Skill       : Machine Learning
Status      : Demonstrated
Evidence/Gap: Trained classification models using Scikit-learn and evaluated them with metrics such as accuracy, precision, recall, and F1-score.
--------------------------------------------------

Overall Recommendation: Recommend proceeding to the next interview stage; candidate shows strong 